# Transformer kernel — pass 4 measurements (Kaggle 2× Tesla T4)

Pass 4 ships opt-in: **dual-GPU batch split** (`--dual-gpu`), **fused Triton GEMM epilogues** (`--fused-out-proj`, `--fused-ffn`), a **materialized-score bmm attention** path for head_dim > 128 (`--attention bmm`), a cuBLASLt tanh-GELU probe (`--gelu-epilogue`), dual-stream graph capture (`--graph-streams 2`), plus the attribution tools `profile_case.py` / `bench_micro.py`. Design + predictions: `docs/pass4-plan.md`.

This notebook also carries the **F1–F4 follow-ups** that pass 3 queued but never ran (the old notebook targeted the since-deleted `kq9ns1` branch).

**Priorities if the session is short**: G0 (gate) → G1 (dual-GPU) → G2 (fused GEMMs) → G3 (case 8) → F1 (regression) → the rest. Every cell is independent after G0; every run writes JSON into `results/`, and the last cells print a summary and pack `results.tar.gz`.

Repo: https://github.com/danielfodgaard/transformer-kernel

In [ ]:
BRANCH = "claude/transformer-kernel-gpu-optimization-4q6ki7"  # pass-4 exploratory branch

!nvidia-smi --query-gpu=index,name,temperature.gpu,clocks.sm --format=csv
!cd /kaggle/working && rm -rf transformer-kernel && \
  git clone -q https://github.com/danielfodgaard/transformer-kernel.git && \
  cd transformer-kernel && git checkout -q {BRANCH} && git log --oneline -1
import torch, triton
print(torch.__version__, '| triton', triton.__version__)
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## G0 — gate: kernel tests, environment probes

Fail-early: the GPU test script now covers the pass-4 kernels (fused GEMM epilogues, causal softmax, `write_sum=False`); the pytest suite adds the dual-GPU wrapper (its two-GPU test only runs here). Then two 1-minute probes the pass-4 designs depend on: **(a)** PCIe D2D bandwidth + peer access (the dual-GPU balance model's input), **(b)** confirmation that the mem-efficient SDPA output makes `transpose(1,2).reshape` a zero-copy view (the pass-4 erratum to the pass-3 transpose claim).

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/test_kernels.py
!cd /kaggle/working/transformer-kernel && python -m pytest src/test_triton_kernels.py src/test_dual_gpu.py -q
!cd /kaggle/working/transformer-kernel && python src/bench_micro.py p2p
import torch, torch.nn.functional as F
q = torch.randn(2, 4, 64, 32, device='cuda', dtype=torch.float16)
ctx = F.scaled_dot_product_attention(q, q, q, is_causal=True, scale=0.1767767)
view = ctx.transpose(1, 2).reshape(2, 64, 128)
print('sdpa out strides', ctx.stride(), '| transpose+reshape zero-copy:', view.data_ptr() == ctx.data_ptr())

## G1 — dual-GPU batch split (the 2×T4 lever)

Balance-model predictions at ~8 GB/s PCIe: case 6 198.6 → ~128 ms (7.35x → ~11.4x), case 8 25.2 → ~14.4 ms, case 13 18.75 → ~11.1 ms; case 14 ~155 → ~80 s. The control sweep runs in the same session so deltas are thermal-drift-free. The 25-trial stress checks the split does not move case 6's fragile margin; the `CUDA_VISIBLE_DEVICES=0` run must reproduce single-GPU numbers exactly (no-op fallback). Calibration prints a `[dual-gpu] calibrated split` line — sanity-check it lands near 0.35–0.45.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 1,5,6,8,13 --out results/pass4-control.json
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6,8,13 --out results/pass4-dual.json -- --dual-gpu --dual-verify
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4-dual-case6-stress.json -- --dual-gpu \
  --accuracy-trials 25 --warmup 1 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 13 --out results/pass4-dual-frac35.json -- --dual-gpu --dual-fraction 0.35
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 13 --out results/pass4-dual-frac45.json -- --dual-gpu --dual-fraction 0.45
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 5 --out results/pass4-dual-case5.json -- --dual-gpu --dual-min-elements 1000000
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 12 --out results/pass4-dual-graphs.json -- --dual-gpu --cuda-graphs
!cd /kaggle/working/transformer-kernel && CUDA_VISIBLE_DEVICES=0 python src/sweep.py --cases 13 --out results/pass4-dual-noop.json -- --dual-gpu
!cd /kaggle/working/transformer-kernel && python src/run_case14.py --skip-accuracy --repeats 3 --dual-gpu --out results/pass4-case14-dual.json

## G2 — fused Triton GEMM epilogues

Corrected traffic model (~4·T·D bytes/layer per fused site; the transpose was already free): case 6 predicted −15–31 ms (→ ~8.0–8.8x), cases 1/5/13 proportionally, graphed cases lose ~12 in-graph kernels. Case 6 first — it kills the direction fastest if Triton's sm_75 GEMM underperforms. The 25-trial × 2-seed stress on case 6 gates any dispatch adoption (different fp32 summation order can move the worst element either way). Watch graphed runs for `[cuda-graphs] capture failed` lines.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4-case6-fusedgemm.json -- --fused-out-proj --fused-ffn
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4-case6-outproj-only.json -- --fused-out-proj
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 1,5,13 --out results/pass4-bw-fusedgemm.json -- --fused-out-proj --fused-ffn
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 2,3,4,12 --out results/pass4-cg-fusedgemm.json -- --cuda-graphs --fused-out-proj --fused-ffn
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 9,10,11 --out results/pass4-heads-fusedgemm.json -- --fused-out-proj --fused-ffn
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4-case6-fusedgemm-stress-a.json -- --fused-out-proj --fused-ffn \
  --accuracy-trials 25 --warmup 1 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4-case6-fusedgemm-stress-b.json -- --fused-out-proj --fused-ffn \
  --accuracy-trials 25 --seed 9999 --warmup 1 --repeats 1 --benchmark-rounds 1

## G3 — case 8 attribution + `--attention bmm`

Where do case 8's 25.2 ms go? The arithmetic says ~11 ms GEMM + ~4 ms LN traffic, leaving ~5–9 ms suspected in the CUTLASS mem-efficient kernel's hd=256 config. E1 profiles one forward per-op; E2/E3 isolate cuBLAS TFLOPS and the SDPA hd cliff; then the bmm path runs the official case 8 plus a 25-trial stress (its fp16 score materialization is a NEW rounding source — the stress is the ship/kill gate; fallback if it fails: fp32 QK^T). Cases 9/11/12 are expected neutral-to-worse — measured to close the question.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/profile_case.py --causal --batch-size 64 --seq-len 128 --d-model 1024 --heads 4 --ffn-dim 1024 --layers 4
!cd /kaggle/working/transformer-kernel && python src/bench_micro.py gemm --shapes 8192x1024x3072,8192x1024x1024
!cd /kaggle/working/transformer-kernel && python src/bench_micro.py sdpa --b 64 --heads 4 --seq 128 --hd-sweep 32,64,128,256 --causal
!cd /kaggle/working/transformer-kernel && python src/bench_micro.py bmmattn --b 64 --heads 4 --seq 128 --hd 256 --causal
!cd /kaggle/working/transformer-kernel && python src/bench_micro.py bmmattn --b 64 --heads 16 --seq 128 --hd 8 --causal
!cd /kaggle/working/transformer-kernel && python src/bench_micro.py bmmattn --b 64 --heads 1 --seq 128 --hd 128 --causal
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 8 --out results/pass4-case8-bmm.json -- --attention bmm
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 8 --out results/pass4-case8-bmm-stress.json -- --attention bmm \
  --accuracy-trials 25 --warmup 1 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 9,11,12 --out results/pass4-bmm-sweep.json -- --attention bmm

## G4 — launch-bound floor probes (measure-first verdicts)

Free probes before any new engineering: a per-kernel trace of a graphed tiny shape (validates the ~37-kernel inventory and per-kernel floor), `--attention math` inside graphs (if 7 kernels/layer tie 1, the floor is latency not work), graphs on cases 9/10 (never measured in pass 2), and the dual-stream capture experiment on the B≥4 graphed shapes.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/profile_case.py --causal --batch-size 4 --seq-len 128 --d-model 128 --heads 4 --ffn-dim 128 --layers 4 --cuda-graphs
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 2,3,12 --out results/pass4-cg-math.json -- --cuda-graphs --attention math
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 9,10 --out results/pass4-cg-heads.json -- --cuda-graphs
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 3,4,12 --out results/pass4-cg-2stream.json -- --cuda-graphs --graph-streams 2

## G5 — cuBLASLt GELU epilogue probe + TunableOp

`--gelu-epilogue` uses the tanh GELU approximation (~1e-3-class deviation): the stress cells decide whether any case can absorb it — do **not** adopt it anywhere that fails. The TunableOp probe is 5 minutes and probably a no-op.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 1,5,6,13 --out results/pass4-gelu.json -- --gelu-epilogue
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4-gelu-case6-stress.json -- --gelu-epilogue \
  --accuracy-trials 25 --warmup 1 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && PYTORCH_TUNABLEOP_ENABLED=1 PYTORCH_TUNABLEOP_TUNING=1 \
  python src/run_case.py --causal --batch-size 64 --seq-len 128 --d-model 1024 --heads 4 --ffn-dim 1024 --layers 4

## F1–F4 — the pass-3 follow-up backlog (unchanged questions, merged tree)

* **F1** best-config regression (`configs/best.json`) — every case should land near the pass-2 table (geomean 7.11x) — plus the case-14 out-of-core quick pass.
* **F2** case 7: plain defaults vs `torch.compile reduce-overhead`, same session (cross-session hint was 1.60 vs 1.46 ms).
* **F3** case 6 seed-robustness pricing: `--fp16-max-elements 100000000` (fp32 dispatch) cost + 25-trial × 2-seed stress.
* **F4** the Triton attention kernel's negative result, reproduced post-merge (if it ever starts winning after a Triton upgrade, this is how we notice).

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --config configs/best.json --skip 14 --out results/pass4f-best-regression.json
!cd /kaggle/working/transformer-kernel && python src/run_case14.py --max-samples 4
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 7 --out results/pass4f-case7-default.json
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 7 --out results/pass4f-case7-compiled.json -- --compile-user --compile-mode reduce-overhead
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4f-case6-fp32dispatch.json -- --fp16-max-elements 100000000
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4f-case6-fp32dispatch-stress-a.json -- --fp16-max-elements 100000000 \
  --accuracy-trials 25 --warmup 1 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 --out results/pass4f-case6-fp32dispatch-stress-b.json -- --fp16-max-elements 100000000 \
  --accuracy-trials 25 --seed 9999 --warmup 1 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 11,13 --out results/pass4f-attn-triton.json -- --attention triton

In [ ]:
# Summary: every pass-4 run vs the pass-2 measured-best table.
import json, pathlib

PASS2_BEST = {1: 7.053, 2: 9.545, 3: 9.186, 4: 6.636, 5: 6.981, 6: 7.348,
              7: 3.932, 8: 5.087, 9: 4.434, 10: 5.244, 11: 10.193,
              12: 6.949, 13: 16.977}

root = pathlib.Path('/kaggle/working/transformer-kernel/results')
for path in sorted(root.glob('pass4*.json')):
    data = json.loads(path.read_text())
    if 'cases' not in data:
        continue  # run_case14 summaries have their own schema
    print(f"\n=== {path.stem} | args={data.get('passthrough_args')}")
    for case in data['cases']:
        cid = case['case']['id']
        acc = case.get('accuracy') or {}
        speed = f"{case['speedup']:.3f}x" if case.get('speedup') else '-'
        max_abs = acc.get('max_abs_error')
        err = f"{max_abs:.2e}" if max_abs is not None else '-'
        ref = f"  (pass-2 best {PASS2_BEST[cid]:.2f}x)" if cid in PASS2_BEST else ''
        print(f"  case {cid:>2} {case['status']:<16} {speed:>9}  max_abs={err}{ref}")

In [ ]:
# After a good session: distill accuracy-passing wins into the dispatch table
# (explicit flags still beat it; dual-gpu/cuda-graphs are provenance-only).
!cd /kaggle/working/transformer-kernel && python src/dispatch.py results/pass4-*.json results/pass4f-*.json --out configs/dispatch.json
!cd /kaggle/working/transformer-kernel && tar czf /kaggle/working/results.tar.gz results/ configs/dispatch.json
print('Download results.tar.gz from the notebook Output panel')